# Week 8 Results

In [2]:
# %% 
import pandas as pd
import numpy as np

# %%
# Load your predictions from last week
pred_lines = pd.read_csv("cfb_betting_lines.csv")

# Load actual results from week 5
week5 = pd.read_csv("cfbweek9.csv")

# %%
# Filter for completed, non-postseason week 8 games
completed_games = week5[
    (week5["Completed"] == True) & 
    (week5["SeasonType"] != "postseason") &
    (week5["Week"] == 9)
][["HomeTeam","AwayTeam","HomePoints","AwayPoints"]].copy()

# Calculate actual margin of victory (home - away)
completed_games["ActualSpread"] = completed_games["HomePoints"] - completed_games["AwayPoints"]

# %%
# Merge with your predicted lines
merged = completed_games.merge(
    pred_lines[["HomeTeam","AwayTeam","Spread","Line"]],
    on=["HomeTeam","AwayTeam"],
    how="left"
)

# %%
# Evaluate how the pick did
def grade_pick(row):
    predicted_spread = row["Spread"]
    actual_spread = row["ActualSpread"]

    if np.isnan(predicted_spread):
        return "No line"
    
    # If predicted spread > 0, you picked Home favorite
    if predicted_spread > 0:
        if actual_spread > 0:  # Home won by enough
            return "Correct"
        else:
            return "Wrong"
    elif predicted_spread < 0:
        if actual_spread < 0:  # Away won/covered
            return "Correct"
        else:
            return "Wrong"
    else:
        # Pick'em case
        if actual_spread == 0:
            return "Push"
        else:
            return "Wrong"

merged["Result"] = merged.apply(grade_pick, axis=1)

# %%
# Summary stats
correct = (merged["Result"]=="Correct").sum()
wrong = (merged["Result"]=="Wrong").sum()
push = (merged["Result"]=="Push").sum()

print(f"Week 5 Performance:")
print(f"Correct Picks: {correct}")
print(f"Wrong Picks:   {wrong}")
print(f"Pushes:        {push}")

# %%
# Save detailed comparison
merged.to_csv("week9_line_performance.csv", index=False)

# Show the game-by-game results
merged


Week 5 Performance:
Correct Picks: 242
Wrong Picks:   65
Pushes:        0


,HomeTeam,AwayTeam,HomePoints,AwayPoints,ActualSpread,Spread,Line,Result
0,Florida International,Kennesaw State,26.0,45.0,-19.0,-15.0,Kennesaw State -15.0,Correct
1,Louisiana Tech,Western Kentucky,27.0,28.0,-1.0,14.5,Louisiana Tech -14.5,Wrong
2,Delaware,Middle Tennessee,31.0,28.0,3.0,12.5,Delaware -12.5,Correct
3,New Mexico State,Missouri State,17.0,24.0,-7.0,-7.0,Missouri State -7.0,Correct
4,Georgia State,South Alabama,31.0,38.0,-7.0,-7.0,South Alabama -7.0,Correct
...,...,...,...,...,...,...,...,...
302,Texas A&M-Kingsville,Western Oregon,27.0,34.0,-7.0,8.0,Texas A&M-Kingsville -8.0,Wrong
303,Pomona Pitzer,La Verne,8.0,26.0,-18.0,21.0,Pomona Pitzer -21.0,Wrong
304,California Lutheran University,Redlands,24.0,30.0,-6.0,2.0,California Lutheran University -2.0,Wrong
305,Chapman,Claremont-Mudd-Scripps College,37.0,31.0,6.0,20.5,Chapman -20.5,Correct


In [4]:
# Difference between your predicted spread and actual result
merged["ATS_Error"] = merged["Spread"] - merged["ActualSpread"]

# Absolute error to judge accuracy regardless of direction
merged["Abs_ATS_Error"] = merged["ATS_Error"].abs()

# Summary statistics
print("Average Error (Bias):", merged["ATS_Error"].mean())
print("Mean Absolute Error (MAE):", merged["Abs_ATS_Error"].mean())
print("Median Absolute Error:", merged["Abs_ATS_Error"].median())

within_3 = (merged["Abs_ATS_Error"] <= 3).mean()
within_7 = (merged["Abs_ATS_Error"] <= 7).mean()

print(f"Within 3 pts: {within_3:.0%}")
print(f"Within 7 pts: {within_7:.0%}")

within_3 = (merged["Abs_ATS_Error"] <= 3).mean()
within_7 = (merged["Abs_ATS_Error"] <= 7).mean()

print(f"Within 3 pts: {within_3:.0%}")
print(f"Within 7 pts: {within_7:.0%}")


Average Error (Bias): -2.8088235294117645
Mean Absolute Error (MAE): 12.52124183006536
Median Absolute Error: 10.0
Within 3 pts: 15%
Within 7 pts: 33%
Within 3 pts: 15%
Within 7 pts: 33%
